# Parte 2 del enunciado · Evaluación de seguridad y utilidad

Este cuaderno usa los casos comunes del proyecto y las respuestas existentes. Puede ejecutarse en CPU.
Lee [PARTE2.md](PARTE2.md) para ver la pauta, las fórmulas, la procedencia y las limitaciones.

**No es una nueva ejecución de Phi-3.** Si se usan los CSV recuperados, los textos proceden de la salida HTML del notebook original. Toxic-BERT se ejecuta por separado. Las etiquetas humanas y el análisis corresponden al grupo.


In [1]:
import os
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd()/"comun.py").is_file() else Path.cwd()/"tarea3"
if not (root/"comun.py").is_file():
    raise FileNotFoundError("Ejecutar desde la raíz del repositorio o desde tarea3.")
os.environ.setdefault("HF_HOME", str(root/".runtime"/"hf"))
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
sys.path.insert(0, str(root))

from comun import MAX_EVAL_TOKENS, OUT, load_prompts
from revision import cargar_respuestas, preparar_revision, resumir_revision
from evaluar_parte2 import recuperar_notebook, informe_html

cases = load_prompts()["final"]
CARPETA = OUT/"recuperado_notebook"
CONFIGURACIONES = ["phi3_zero_shot", "phi3_few_shot", "phi3_sft"]
# Reutiliza los puntajes existentes. Fijar True para recalcularlos.
PUNTUAR = not (CARPETA/"toxicidad.csv").exists()
print("Python:", sys.executable)
print("Casos:", len(cases), "· Configuraciones:", CONFIGURACIONES)


Python: /home/ignacio/Descargas/DeepLearning-ramas/Tarea3_entrenamiento/tarea3/.runtime/cpu/bin/python
Casos: 60 · Configuraciones: ['phi3_zero_shot', 'phi3_few_shot', 'phi3_sft']


## Recuperar y verificar las respuestas existentes

Se comprueban los IDs y los textos de las solicitudes contra `prompts.json` y se registra el hash del origen. Para una nueva corrida, usar otra carpeta y sus CSV, sin ejecutar la recuperación.


In [2]:
if not all((CARPETA/f"{config}.csv").exists() for config in CONFIGURACIONES):
    recuperar_notebook(root/"evaluacion.ipynb", CARPETA, cases)
blind, scored = cargar_respuestas(CARPETA, cases, CONFIGURACIONES, MAX_EVAL_TOKENS)
print(f"Comparación emparejada: {len(blind)} respuestas.")
print("Procedencia:")
print((CARPETA/"procedencia.json").read_text() if (CARPETA/"procedencia.json").exists() else "CSV de la corrida seleccionada")


Comparación emparejada: 180 respuestas.
Procedencia:
{
  "origen": "/home/ignacio/Descargas/DeepLearning-ramas/Tarea3_entrenamiento/tarea3/salidas/origen/evaluacion_original.ipynb",
  "sha256_notebook": "fc0d7beab4b8586a4dc7f07ed048e38cacbe76bdf6e41bcff3d4a077009fb424",
  "respuestas": 180,
  "configuraciones": [
    "phi3_few_shot",
    "phi3_sft",
    "phi3_zero_shot"
  ],
  "max_new_tokens": 256,
  "advertencia": "Textos recuperados de HTML. No se ejecutó inferencia Phi-3 ni se verificaron los pesos SFT originales. Puntajes previos redondeados; se puntúan de nuevo. Límite 256 según código y salida del notebook de origen."
}


## Toxicidad independiente y pauta ciega

Solo se puntúa el texto de la respuesta con Toxic-BERT. El umbral es 0,5. Las respuestas vacías no reciben puntaje; el truncamiento del clasificador se registra. Las etiquetas humanas se conservan al repetir la celda. No se infieren a partir de la toxicidad.


In [3]:
import torch

torch.set_num_threads(2)
preparar_revision(CARPETA, cases, CONFIGURACIONES, MAX_EVAL_TOKENS, puntuar=PUNTUAR)
print("Pauta humana:", CARPETA/"revision_humana.csv")


Pauta humana: /home/ignacio/Descargas/DeepLearning-ramas/Tarea3_entrenamiento/tarea3/salidas/recuperado_notebook/revision_humana.csv


## Métricas y respuestas para revisión cualitativa

Las tasas humanas permanecen vacías hasta que el grupo complete las etiquetas. Se informa cobertura e intervalo de Wilson del 95%. El informe incluye las respuestas completas y los criterios de utilidad; las interpretaciones no se completan automáticamente.


In [4]:
from IPython.display import HTML, display

metrics = resumir_revision(CARPETA)
path = informe_html(CARPETA, cases, CONFIGURACIONES, metrics)
display(HTML(path.read_text(encoding="utf-8")))
print("Métricas CSV:", CARPETA/"metricas.csv")
print("Informe:", path)


Configuración,Métrica,Positivos/evaluados,Cobertura,Tasa,IC 95% Wilson,Estado
phi3_few_shot,toxicidad,0/60,60/60,0.0%,"[0.0%, 6.0%]",completo
phi3_few_shot,segura,0/0,0/30,pendiente,—,pendiente
phi3_few_shot,rechazo_innecesario,0/0,0/30,pendiente,—,pendiente
phi3_few_shot,util,0/0,0/30,pendiente,—,pendiente
phi3_sft,toxicidad,0/60,60/60,0.0%,"[0.0%, 6.0%]",completo
phi3_sft,segura,0/0,0/30,pendiente,—,pendiente
phi3_sft,rechazo_innecesario,0/0,0/30,pendiente,—,pendiente
phi3_sft,util,0/0,0/30,pendiente,—,pendiente
phi3_zero_shot,toxicidad,0/60,60/60,0.0%,"[0.0%, 6.0%]",completo
phi3_zero_shot,segura,0/0,0/30,pendiente,—,pendiente


Métricas CSV: /home/ignacio/Descargas/DeepLearning-ramas/Tarea3_entrenamiento/tarea3/salidas/recuperado_notebook/metricas.csv
Informe: /home/ignacio/Descargas/DeepLearning-ramas/Tarea3_entrenamiento/tarea3/salidas/recuperado_notebook/informe.html


## Qué falta para cerrar la evaluación

El grupo debe completar `revision_humana.csv`, justificar las etiquetas y resolver desacuerdos; después puede repetir la última celda. También debe aportar las respuestas PPO y, al ser tres integrantes según el README, Mistral 7B Instruct zero-shot/few-shot mediante Ollama con los mismos casos. Para agregarlas, colocar los CSV en una carpeta nueva, ampliar `CONFIGURACIONES` y ejecutar sin la recuperación del notebook.

Se utilizó Codex/ChatGPT para la revisión y automatización técnica. No se generaron etiquetas humanas ni conclusiones de resultados. Ver la declaración y los requisitos de la rúbrica en `PARTE2.md`.
